# Netflix Intelligence — Text-to-SQL LLM App

Ask questions about a catalog of 8,800+ Netflix titles in plain language. The app translates each question into SQL (OpenAI gpt-4o-mini via LangChain), runs it on Google BigQuery, and answers conversationally through a Gradio web interface, with persistent conversation memory.

See `README.md` for the business case, architecture and design decisions.

> **Setup:** add `OPENAI_API_KEY` to Colab Secrets (key icon in the left sidebar) and have access to a BigQuery project with the `netflix_titles` table.

## 1. Install dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community google-cloud-bigquery pandas gradio

## 2. Credentials
The OpenAI key is read from Colab Secrets — it is never written in the code.

In [ ]:
from google.colab import userdata
import os

api_key = userdata.get('OPENAI_API_KEY')

if api_key:
    os.environ["OPENAI_API_KEY"] = api_key
    print("✓ API key de OpenAI cargada correctamente")
else:
    print("✗ Error: no se encontró OPENAI_API_KEY en los secretos")

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("✓ Autenticación con Google Cloud lista")

## 3. Text-to-SQL engine
The prompt gives the model the table schema plus explicit rules for this dataset (multi-value columns, exact `type` values, answer shapes).

In [ ]:
from langchain_openai import ChatOpenAI

def generate_sql(user_question: str) -> str:
    model = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )
    prompt = f"""
    Eres un experto en SQL y BigQuery.

    Tienes esta tabla:
    Tabla: `netflix-intelligence-497821.netflix_dataset.netflix_titles`

    Columnas y valores importantes:
    - show_id: identificador unico
    - type: solo tiene dos valores exactos: 'Movie' o 'TV Show'
    - title: nombre del contenido
    - director: nombre del director (puede ser nulo)
    - cast: actores separados por coma (puede ser nulo)
    - country: pais de produccion, puede tener multiples paises separados por coma
    - date_added: fecha en que se agrego a Netflix
    - release_year: año de estreno, es un numero entero
    - rating: clasificacion de edad (TV-MA, PG-13, R, G, etc)
    - duration: '90 min' para peliculas o '2 Seasons' para series
    - listed_in: generos separados por coma. Ejemplos: 'Documentaries', 'Comedies', 'Dramas', 'Action & Adventure', 'International Movies'
    - description: sinopsis del contenido

    REGLAS IMPORTANTES:
    - Para buscar generos usa: listed_in LIKE '%genero%'
    - Para buscar paises usa: country LIKE '%pais%'
    - Para buscar actores usa: cast LIKE '%nombre%'
    - type solo acepta 'Movie' o 'TV Show'
    - Si el usuario pregunta CUANTOS hay, usa SELECT COUNT(*) as total
    - Si el usuario pregunta QUE hay o pide lista, usa SELECT title, type, listed_in LIMIT 20
    - Si el usuario pregunta por UN titulo especifico, usa SELECT * LIMIT 1

    El usuario pregunta: {user_question}

    Devuelve SOLO el SQL, sin explicaciones, sin markdown, sin comillas.
    """
    response = model.invoke(prompt)
    return response.content

print("✓ Función generate_sql lista")

## 4. Query execution on BigQuery
Guardrails: only read-only `SELECT` queries are executed, and non-aggregate results are capped at 20 rows.

In [ ]:
from google.cloud import bigquery

def run_query(sql: str) -> str:
    client = bigquery.Client(project="netflix-intelligence-497821")

    # Limpiar posibles bloques de markdown que devuelva el modelo
    sql = sql.strip().removeprefix("```sql").removeprefix("```").removesuffix("```").strip()

    # Guardrail: solo se permiten consultas de lectura
    if not sql.upper().startswith(("SELECT", "WITH")):
        return "Consulta bloqueada: solo se permiten consultas de lectura (SELECT)."

    # Solo agregar LIMIT si no es COUNT y no tiene LIMIT ya
    if "COUNT" not in sql.upper() and "LIMIT" not in sql.upper():
        sql = sql + " LIMIT 20"

    results = client.query(sql).to_dataframe()
    return results.to_string(index=False)

print("✓ Función run_query lista")

## 5. Conversational layer with memory
`SQLChatMessageHistory` stores each session so the assistant can handle follow-up questions.

In [ ]:
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
import datetime

def get_response(user_question: str, session_id: str = "default") -> str:

    # Abrir el cuaderno de memoria
    history = SQLChatMessageHistory(
        session_id=session_id,
        connection="sqlite:///conversations.db"
    )

    # Guardar la pregunta del usuario
    history.add_message(HumanMessage(content=user_question))

    # Generar SQL y ejecutar
    sql = generate_sql(user_question)
    raw_results = run_query(sql)

    # Construir contexto con historial completo
    mensajes_anteriores = ""
    for msg in history.get_messages()[:-1]:
        if isinstance(msg, HumanMessage):
            mensajes_anteriores += f"Usuario: {msg.content}\n"
        elif isinstance(msg, AIMessage):
            mensajes_anteriores += f"Asistente: {msg.content}\n"

    model = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )

    prompt = f"""
    Eres un asistente experto en el catalogo de Netflix.

    Historial de la conversacion:
    {mensajes_anteriores}

    Pregunta actual: {user_question}

    Resultados de la base de datos:
    {raw_results}

    INSTRUCCIONES:
    - Si los resultados son un numero (COUNT), responde con ese numero exacto.
    - Si los resultados son una lista de titulos, presenta esa lista de forma clara.
    - Si los resultados estan vacios, dilo claramente.
    - Usa el historial para entender el contexto de la conversacion.
    - Responde SOLO lo que el usuario pregunto.
    """

    response = model.invoke(prompt)
    respuesta = response.content

    # Guardar la respuesta en memoria
    history.add_message(AIMessage(content=respuesta))

    return respuesta

print("✓ Función get_response con memoria lista")

## 6. Test: question + contextual follow-up

In [ ]:
print(get_response("¿Qué documentales hay disponibles?"))

Aquí tienes una lista de documentales disponibles en Netflix:

1. Ghost of the Mountains
2. Growing Up Wild
3. The Pixar Story
4. Océans
5. Justin Bieber: Never Say Never
6. (T)ERROR
7. After Porn Ends
8. APEX: The Story of the Hypercar
9. Fishtail
10. Harry Benson: Shoot First
11. Holy Hell
12. Miss Representation
13. Newtown
14. Prescription Thugs
15. Raiders!: The Story of the Greatest Fan Film Ever Made
16. Rats
17. Sour Grapes
18. Democrats
19. How to Change the World
20. I Am Sun Mu


In [ ]:
print(get_response("¿Cuál de esos me recomiendas para ver con mis hijos?"))

De la lista de documentales que te proporcioné, te recomiendo "Growing Up Wild". Es un documental que muestra la vida de varios animales jóvenes en su hábitat natural, lo que puede ser interesante y educativo para tus hijos.


## 7. Web interface (Gradio)
`share=True` publishes a temporary public URL for demos.

In [ ]:
import gradio as gr
import datetime

def chatbot(user_question, session_id):
    if not user_question.strip():
        return "Por favor escribe una pregunta."
    respuesta = get_response(user_question, session_id=session_id)
    return respuesta

session_id = f"session-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

interfaz = gr.Interface(
    fn=lambda q: chatbot(q, session_id),
    inputs=gr.Textbox(
        placeholder="¿Cuántas películas de India hay en Netflix?",
        label="Tu pregunta"
    ),
    outputs=gr.Textbox(
        label="Respuesta",
        lines=15
    ),
    title="Netflix Intelligence",
    description="Haz preguntas en lenguaje natural sobre el catálogo de Netflix."
)

interfaz.launch(share=True)